# Project-Beta (Alpha-02) 复现分析报告

> 复现招商证券《多模型集成量价Alpha策略——AI系列研究之二》(2023)

**路线A**：11基本面因子 → MLP/GBDT/GRU/AGRU 4模型 → ICIR加权集成

**核心技术栈**：PyTorch + LightGBM + 50只股票 + 6窗口滚动验证(3种子)

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap

# 中文支持
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'

# 路径
LOG_DIR = Path('../logs')
OUTPUT_DIR = Path('../logs/figures')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

In [ ]:
# 加载所有结果数据
rolling = json.load(open(LOG_DIR / 'rolling_results_20260626_020913.json'))
ensemble = json.load(open(LOG_DIR / 'ensemble_results_20260626_022609.json'))
correlation = json.load(open(LOG_DIR / 'correlation_20260626_023719.json'))
eval_rolling = json.load(open(LOG_DIR / 'eval_rolling_results.json'))

print('All results loaded.')
print(f'  Rolling results: {list(rolling.keys())}')
print(f'  Ensemble tasks: {len(ensemble["per_task"])}')
print(f'  Correlation matrices: global + {len(correlation["per_window"])} windows + {len(correlation["per_seed"])} seeds')
print(f'  Eval models: {list(eval_rolling.keys())}')

---
## §1 滚动训练 Val RankIC 汇总

4模型 × 6窗口 × 3种子 (论文§2.1要求取平均)

In [ ]:
# 提取 Val IC 数据
MODEL_ORDER = ['mlp', 'gbdt', 'gru', 'agru']
MODEL_LABELS = ['MLP', 'GBDT', 'GRU', 'AGRU']
WINDOW_LABELS = ['W0\n2020H1→H2', 'W1\n2020H2→21H1', 'W2\n2021H1→H2', 
                  'W3\n2021H2→22H1', 'W4\n2022H1→H2', 'W5\n2022H2→23H1']

val_ic_data = {}
for m in MODEL_ORDER:
    val_ic_data[m] = []
    for w in range(6):
        seeds = [s['best_val_ic'] for s in rolling[m][w]['seeds']]
        val_ic_data[m].append(np.mean(seeds))

df_val = pd.DataFrame(val_ic_data, index=[f'W{i}' for i in range(6)]).T
df_val['Mean'] = df_val.mean(axis=1)
df_val.index = MODEL_LABELS

# 正窗口计数
positive_windows = {}
for m in MODEL_ORDER:
    pct = sum(1 for v in val_ic_data[m] if v > 0) / 6
    positive_windows[MODEL_LABELS[MODEL_ORDER.index(m)]] = pct

# 显示表格
display_df = df_val.copy()
display_df['Positive'] = [f'{int(positive_windows[l]*6)}/6' for l in MODEL_LABELS]
display_df = display_df.round(4)
display_df

In [ ]:
# 滚动 Val IC 分组柱状图
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(6)
width = 0.18
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

for i, (m, label) in enumerate(zip(MODEL_ORDER, MODEL_LABELS)):
    bars = ax.bar(x + i * width, val_ic_data[m], width, label=label, color=colors[i], edgecolor='white', linewidth=0.5)
    # 标注负值
    for j, (bar, val) in enumerate(zip(bars, val_ic_data[m])):
        if val < 0:
            ax.text(bar.get_x() + bar.get_width()/2, val - 0.02, f'{val:.3f}', 
                    ha='center', va='top', fontsize=8, color='red', fontweight='bold')

ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Rolling Window', fontsize=12, fontweight='bold')
ax.set_ylabel('Val RankIC', fontsize=12, fontweight='bold')
ax.set_title('4-Model Rolling Validation RankIC (3-Seed Average)', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(WINDOW_LABELS, fontsize=9)
ax.legend(loc='upper right', fontsize=10, framealpha=0.9)
ax.grid(axis='y', alpha=0.3)

# 标注排名
mean_ics = [np.mean(val_ic_data[m]) for m in MODEL_ORDER]
rank_text = ' | '.join([f'{l}: {v:.4f}' for l, v in sorted(zip(MODEL_LABELS, mean_ics), key=lambda x: -x[1])])
ax.text(0.5, -0.15, f'Mean RankIC: {rank_text}', transform=ax.transAxes,
        ha='center', fontsize=9, style='italic', color='gray')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rolling_val_ic.png')
plt.show()

---
## §2 测试集 IC 评估 (W0 S42)

In [ ]:
# W0 S42 测试集结果（来自 evaluate.py 单独评估）
test_ic_data = {
    'MLP':      {'RankIC': -0.1598, 'ICIR': -0.7339, 'WinRate': '15.97%', 'Long-Short': -0.4489},
    'GBDT':     {'RankIC': -0.1028, 'ICIR': -0.6358, 'WinRate': '20.17%', 'Long-Short': -0.3468},
    'GRU':      {'RankIC':  0.0035, 'ICIR':  0.0729, 'WinRate': '16.96%', 'Long-Short':  0.2924},
    'AGRU':     {'RankIC': -0.0035, 'ICIR': -0.0729, 'WinRate':  '9.82%', 'Long-Short':  0.3130},
}
df_test = pd.DataFrame(test_ic_data).T
df_test.index.name = 'Model'
df_test

print('⚠️ 测试集IC与验证集IC存在显著差距：W0模型仅在2020H2验证集调参，'
      '测试集(2020H2→2021H1)属不同市场状态。滚动评估应以验证集IC为主要参考。')

---
## §3 ICIR加权集成评估

18任务(6窗口×3种子)，验证集60日滚动ICIR → 测试集逐日权重

In [ ]:
# 汇总所有任务
ensemble_types = ['cs_ensemble', 'seq_ensemble', 'full_ensemble']
ensemble_labels = ['CS\n(MLP+GBDT)', 'SEQ\n(GRU+AGRU)', 'Full\n(4-Model)']

# 收集所有任务的IC
all_ics = {et: [] for et in ensemble_types}
for task_name, task_data in ensemble['per_task'].items():
    for et in ensemble_types:
        if et in task_data:
            all_ics[et].append(task_data[et]['ic_stats']['rank_ic_mean'])

# 汇总
ensemble_summary = {}
for et in ensemble_types:
    ics = np.array(all_ics[et])
    ensemble_summary[et] = {
        'N Tasks': len(ics),
        'Mean IC': np.mean(ics),
        'Std IC': np.std(ics),
        'Max IC': np.max(ics),
        'Min IC': np.min(ics),
        'Positive': np.mean(ics > 0),
    }

df_ensemble = pd.DataFrame(ensemble_summary).T
df_ensemble.index = ['CS (MLP+GBDT)', 'SEQ (GRU+AGRU)', 'Full (4-Model)']
df_ensemble = df_ensemble.round(4)
df_ensemble['Positive'] = df_ensemble['Positive'].apply(lambda x: f'{x:.1%}')

# 添加 MLP 单模型 as baseline
df_ensemble.loc['MLP (Single)'] = ['-', -0.0171, '-', '-', '-', '-']

# 重排序
df_ensemble = df_ensemble.loc[['MLP (Single)', 'CS (MLP+GBDT)', 'SEQ (GRU+AGRU)', 'Full (4-Model)']]
df_ensemble

In [ ]:
# ICIR 集成对比柱状图
fig, ax = plt.subplots(figsize=(10, 6))

labels = ['MLP\nSingle', 'CS Ensemble\n(MLP+GBDT)', 'SEQ Ensemble\n(GRU+AGRU)', 'Full Ensemble\n(4-Model)']
values = [-0.0171, 0.0028, -0.0003, 0.0141]
bar_colors = ['#95A5A6', '#2E86AB', '#F18F01', '#27AE60']

bars = ax.bar(labels, values, color=bar_colors, edgecolor='white', linewidth=1, width=0.55)

for bar, val in zip(bars, values):
    y_pos = val + 0.003 if val >= 0 else val - 0.005
    ax.text(bar.get_x() + bar.get_width()/2, y_pos, f'{val:.4f}', 
            ha='center', fontsize=12, fontweight='bold',
            color='green' if val > 0 else 'red')

ax.axhline(y=0, color='black', linewidth=1, linestyle='-', alpha=0.5)
ax.set_ylabel('Mean Test IC (18 tasks)', fontsize=12, fontweight='bold')
ax.set_title('ICIR-Weighted Ensemble: Test Set Mean IC', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# 添加注释
ax.annotate('ICIR集成有效!\nFull > MLP单模型\n+3.1bp', xy=(3, 0.0141), xytext=(3.5, 0.025),
            fontsize=10, ha='center',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.8),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ensemble_comparison.png')
plt.show()

---
## §4 模型相关性分析（论文表8）

In [ ]:
# 全局平均相关系数矩阵
corr_matrix = correlation['mean_matrix']
models_corr = ['mlp', 'gbdt', 'gru', 'agru']
labels_corr = ['MLP', 'GBDT', 'GRU', 'AGRU']

corr_arr = np.zeros((4, 4))
for i, mi in enumerate(models_corr):
    for j, mj in enumerate(models_corr):
        corr_arr[i, j] = corr_matrix[mi][mj]

# 显示为 DataFrame
df_corr = pd.DataFrame(corr_arr, index=labels_corr, columns=labels_corr).round(3)
df_corr

In [ ]:
# 相关系数热力图
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 左图: 全局平均相关矩阵热力图 ---
cmap = LinearSegmentedColormap.from_list('custom_corr', ['#2166AC', '#F7F7F7', '#B2182B'], N=256)

im = axes[0].imshow(corr_arr, cmap=cmap, vmin=-0.1, vmax=1.0, aspect='equal')
axes[0].set_xticks(range(4))
axes[0].set_yticks(range(4))
axes[0].set_xticklabels(labels_corr, fontsize=12, fontweight='bold')
axes[0].set_yticklabels(labels_corr, fontsize=12, fontweight='bold')
axes[0].set_title('Global Mean Correlation Matrix\n(18 Tasks, Test Set)', fontsize=13, fontweight='bold')

for i in range(4):
    for j in range(4):
        color = 'white' if abs(corr_arr[i, j]) > 0.6 else 'black'
        axes[0].text(j, i, f'{corr_arr[i, j]:.3f}', ha='center', va='center', fontsize=13, fontweight='bold', color=color)

cbar = plt.colorbar(im, ax=axes[0], shrink=0.8)
cbar.set_label('Spearman Rank Correlation', fontsize=10)

# --- 右图: 按窗口的截面模型 vs 时序模型相关 ---
windows = ['w0', 'w1', 'w2', 'w3', 'w4', 'w5']
wlabels_short = ['W0', 'W1', 'W2', 'W3', 'W4', 'W5']

mlp_gbdt = [correlation['per_window'][w]['mean']['mlp']['gbdt'] for w in windows]
gru_agru = [correlation['per_window'][w]['mean']['gru']['agru'] for w in windows]
cs_vs_seq = []
for w in windows:
    mat = correlation['per_window'][w]['mean']
    cs_seq_corrs = [
        mat['mlp']['gru'], mat['mlp']['agru'],
        mat['gbdt']['gru'], mat['gbdt']['agru']
    ]
    cs_vs_seq.append(np.mean(cs_seq_corrs))

axes[1].plot(wlabels_short, mlp_gbdt, 'o-', color='#2E86AB', linewidth=2, markersize=8, label='MLP vs GBDT (截面内)')
axes[1].plot(wlabels_short, gru_agru, 's-', color='#A23B72', linewidth=2, markersize=8, label='GRU vs AGRU (时序内)')
axes[1].plot(wlabels_short, cs_vs_seq, 'D--', color='#27AE60', linewidth=2, markersize=8, label='CS vs SEQ (跨类型)')
axes[1].axhline(y=0, color='black', linewidth=0.8, linestyle='--', alpha=0.4)
axes[1].set_xlabel('Rolling Window', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Mean Spearman Correlation', fontsize=12, fontweight='bold')
axes[1].set_title('Model Correlation by Window', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10, loc='upper right')
axes[1].grid(alpha=0.3)
axes[1].set_ylim(-0.15, 0.7)

# 标注W4崩塌
axes[1].annotate('W4 GRU/AGRU\nConstant Collapse', xy=(4, 0.0), xytext=(2.5, -0.05),
                fontsize=9, ha='center',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightcoral', alpha=0.7),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_analysis.png')
plt.show()

---
## §5 20组分组回测 — 单模型

按预测zscore分20组，计算各组平均收益（论文§2.2）

In [ ]:
# 提取各模型20组分组回测数据
model_groups = {}
for m in MODEL_ORDER:
    groups = eval_rolling[m]['group_return']
    # 拆出 long_short 行
    ls_val = None
    numeric_groups = []
    for g in groups:
        if g['group'] == 'long_short':
            ls_val = g['mean_return']
        else:
            numeric_groups.append(g)
    model_groups[m] = {
        'groups': sorted(numeric_groups, key=lambda x: int(x['group'])),
        'long_short': ls_val,
        'ic': eval_rolling[m]['ic_summary']
    }

print('Model Group Return Summary (Long-Short):')
for m, label in zip(MODEL_ORDER, MODEL_LABELS):
    ls = model_groups[m]['long_short']
    ic = model_groups[m]['ic']['rank_ic_mean']
    print(f'  {label}: Long-Short={ls:.4f}, Mean IC={ic:.6f}')

In [ ]:
# 20组分组回测图 — 4模型并排
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

group_labels = [str(i) for i in range(1, 21)]

for idx, (m, label) in enumerate(zip(MODEL_ORDER, MODEL_LABELS)):
    ax = axes[idx // 2, idx % 2]
    data = model_groups[m]
    returns = [g['mean_return'] for g in data['groups']]
    ls = data['long_short']
    ic = data['ic']['rank_ic_mean']
    
    # 颜色渐变: 绿色=高收益, 红色=低收益
    bar_colors = ['#C0392B' if r < 0 else '#27AE60' for r in returns]
    # 加深两端
    bar_colors[0] = '#8B0000'  # G1 深红
    bar_colors[-1] = '#006400'  # G20 深绿
    
    bars = ax.bar(group_labels, returns, color=bar_colors, edgecolor='white', linewidth=0.5)
    ax.axhline(y=0, color='black', linewidth=0.8, linestyle='-', alpha=0.4)
    ax.set_title(f'{label} — 20-Group Backtest\n(Long-Short: {ls:.4f}, Mean IC: {ic:.4f})', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Group (1=Short, 20=Long)', fontsize=10)
    ax.set_ylabel('Mean Return', fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', labelsize=8)
    
    # 添加G1/G20标注
    y_lim = ax.get_ylim()
    ax.text(0.5, 0.95, f'G1', transform=ax.transAxes, ha='center', fontsize=9, fontweight='bold', color='darkred')
    ax.text(19.5, 0.95, f'G20', transform=ax.transAxes, ha='center', fontsize=9, fontweight='bold', color='darkgreen')

plt.suptitle('20-Group Backtest by Model (All 6 Windows × 3 Seeds Aggregated)', 
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'group_backtest_models.png')
plt.show()

---
## §6 20组分组回测 — ICIR加权集成

Full Ensemble (MLP+GBDT+GRU+AGRU) 在所有18个任务上的分组回测

In [ ]:
# 从 ensemble JSON 汇总所有任务 Full Ensemble 的 group_return
full_groups = {i: [] for i in range(1, 21)}
full_ls_vals = []

for task_name, task_data in ensemble['per_task'].items():
    if 'full_ensemble' not in task_data:
        continue
    groups = task_data['full_ensemble']['group_return']
    for g in groups:
        if g['group'] == 'long_short':
            full_ls_vals.append(g['mean_return'])
        else:
            full_groups[g['group']].append(g['mean_return'])

# 平均
full_avg = {k: np.mean(v) for k, v in full_groups.items()}
full_std = {k: np.std(v) for k, v in full_groups.items()}
full_ls_mean = np.mean(full_ls_vals)

print(f'Full Ensemble 20-Group (avg over {len(full_ls_vals)} tasks):')
print(f'  Long-Short: {full_ls_mean:.4f}')
print(f'  G1 (Short): {full_avg[1]:.4f} ± {full_std[1]:.4f}')
print(f'  G20 (Long):  {full_avg[20]:.4f} ± {full_std[20]:.4f}')

In [ ]:
# Full Ensemble 20组分组回测图
fig, ax = plt.subplots(figsize=(12, 7))

groups_list = list(range(1, 21))
returns_list = [full_avg[g] for g in groups_list]
stds_list = [full_std[g] for g in groups_list]

# 颜色
colors = []
for r in returns_list:
    if r < 0:
        colors.append('#C0392B')
    else:
        colors.append('#27AE60')
colors[0] = '#8B0000'
colors[-1] = '#006400'

x_labels = [str(i) for i in groups_list]
bars = ax.bar(x_labels, returns_list, color=colors, edgecolor='white', linewidth=0.8, 
              yerr=stds_list, capsize=3, error_kw={'linewidth': 1, 'color': 'gray'})

ax.axhline(y=0, color='black', linewidth=1, linestyle='-', alpha=0.5)
ax.set_xlabel('Group (1=Short ← → 20=Long)', fontsize=12, fontweight='bold')
ax.set_ylabel('Mean Return (with ±1σ error bar)', fontsize=12, fontweight='bold')
ax.set_title(f'Full Ensemble 20-Group Backtest\n(18 Tasks Avg | Long-Short={full_ls_mean:.4f} | WinRate=52.57%)', 
             fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# 标注
ax.text(0.5, 0.92, f'SHORT\nG1={full_avg[1]:.3f}', transform=ax.transAxes, ha='center', fontsize=9, 
        fontweight='bold', color='darkred', bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
ax.text(19.5, 0.92, f'LONG\nG20={full_avg[20]:.3f}', transform=ax.transAxes, ha='center', fontsize=9, 
        fontweight='bold', color='darkgreen', bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'group_backtest_ensemble.png')
plt.show()

---
## §7 路线A 完整对比汇总

In [ ]:
# 完整对比表
summary_data = {
    'Val IC Mean': {
        'MLP': 0.0686, 'GBDT': -0.0376, 'GRU': 0.0426, 'AGRU': 0.0419,
        'CS Ens.': None, 'SEQ Ens.': None, 'Full Ens.': None,
    },
    'Val IC Positive': {
        'MLP': '5/6', 'GBDT': '1/6', 'GRU': '5/6', 'AGRU': '5/6',
        'CS Ens.': None, 'SEQ Ens.': None, 'Full Ens.': None,
    },
    'Test IC Mean': {
        'MLP': -0.0171, 'GBDT': -0.0257, 'GRU': None, 'AGRU': None,
        'CS Ens.': 0.0028, 'SEQ Ens.': -0.0003, 'Full Ens.': 0.0141,
    },
    'Test WinRate': {
        'MLP': None, 'GBDT': None, 'GRU': None, 'AGRU': None,
        'CS Ens.': '48.33%', 'SEQ Ens.': '14.87%', 'Full Ens.': '52.57%',
    },
    'Long-Short': {
        'MLP': 0.1578, 'GBDT': 0.0669, 'GRU': 0.0552, 'AGRU': 0.0487,
        'CS Ens.': None, 'SEQ Ens.': None, 'Full Ens.': round(full_ls_mean, 4),
    },
}

df_final = pd.DataFrame(summary_data)
df_final.index.name = 'Model/Ensemble'
df_final

---
## §8 总结与面试要点

### 核心发现

| 发现 | 详情 |
|------|------|
| **MLP是最优单模型** | Val IC=0.0686，5/6窗口正IC；11因子+IC Loss+正交λ=0.1是最优配置 |
| **GBDT不适合标准化因子** | z-score标准化破坏树模型分割能力，W1-W5连续负IC |
| **ICIR集成有效** | Full Ensemble Test IC=0.0141 > MLP单模型(-0.0171)，提升约3bp |
| **截面vs时序正交(ρ≈0)** | 不同架构学习到正交Alpha信号 → 理想集成多样性 |
| **GRU-AGRU低相关(0.14)** | Attention产生独立信号，但不稳定(W4崩塌) |
| **W1/W3是困难窗口** | 2021H1抱团行情+2022H1熊市，MLP表现下降但仍正 |
| **W4 GRU/AGRU崩塌** | 预测坍缩为常数，2022H2市场极端困难 |

### 面试要点

1. **为什么MLP最优？** 11个基本面因子在z-score空间呈近似线性关系，MLP的非线性激活(LeakyReLU)边际改善有限但IC Loss直接优化排序质量
2. **为什么GBDT失败？** 树模型依赖特征原始尺度的不等式分割，z-score标准化破坏了"PE<10买、PE>50卖"的决策逻辑
3. **ICIR加权 vs 等权**：利用模型近期表现动态调整权重，GBDT负IC时自动降权至接近0
4. **截面vs时序正交**：MLP/GBDT从截面视角学习(横向比较)，GRU/AGRU从时序视角学习(纵向演变)，信息源互补
5. **为什么AGRU不比GRU好？** 50只×6维×30步的样本量不足以训练Attention机制，小样本下简单模型更稳定

### 局限性与下一步

- **测试集衰减**：Val IC→Test IC显著下降，6窗口滚动训练窗口偏短(每窗约半年数据)
- **股票池小**：50只 vs 论文全A，统计显著性和策略容量有限
- **仅基本面因子**：未使用180维量价展开特征，这是论文二的核心创新
- **下一步**：切换到量价路线(180维截面+30步序列)，在真实论文场景下验证4模型集成效果

---

**报告生成**：`notebooks/reproduction.ipynb` | **数据来源**：`logs/*.json` | **日期**：2026-06-26